# Proyecto analítico AquaLimpia

**Asignatura:** Ciencia de Datos  
**Unidad:** 3  
**Estudiante:** Fabiola Arrué Ravanal  

## Propósito del proyecto

Desarrollar un análisis reproducible de los datos operacionales y ambientales de AquaLimpia, evaluar su calidad y generar resultados diferenciados para las áreas de Operaciones y Gestión Ambiental.

## 1. Planteamiento analítico de la solución

AquaLimpia S. A. necesita describir y comparar el desempeño observado de sus plantas para identificar patrones, diferencias y situaciones que requieran revisión. Para responder a esta necesidad se propone un análisis descriptivo, exploratorio y comparativo, desarrollado mediante un flujo reproducible en Python.

### Objetivo general

Describir y comparar el desempeño operacional y ambiental observado de las plantas de AquaLimpia S. A., mediante el análisis reproducible de las variables disponibles, con el propósito de identificar diferencias, asociaciones, patrones temporales y registros que requieran revisión.

### Preguntas de investigación

1. ¿Qué diferencias presentan las plantas en la eficiencia de remoción de demanda biológica de oxígeno (DBO), la DBO de salida, la generación específica de lodos y el cumplimiento registrado?
2. ¿Qué relación se observa entre la DBO de entrada y la DBO de salida, y entre el caudal de entrada y la DBO de salida?
3. ¿Qué asociación existe entre el caudal de entrada y la energía de aireación en cada planta?
4. ¿Qué patrones temporales, candidatos atípicos y limitaciones de calidad deben considerarse antes de utilizar los resultados para apoyar decisiones?

El análisis considera el caudal de entrada, la DBO de entrada y salida, los sólidos suspendidos totales (SST), el pH, la eficiencia de remoción, la energía de aireación, la generación de lodos y el cumplimiento registrado en el dataset. Los resultados se presentan mediante visualizaciones interactivas y reportes diferenciados para Operaciones y Gestión Ambiental.

No se desarrolla una solución predictiva orientada a anticipar riesgos regulatorios, porque `cumplimiento_norma` no documenta el criterio utilizado y cinco valores de DBO de salida aparecen asociados con ambas clasificaciones. A ello se suman la cobertura temporal limitada y la ausencia de variables operacionales y contextuales necesarias para sustentar una predicción confiable. El análisis tampoco permite establecer causalidad ni verificar cumplimiento legal. Su alcance consiste en organizar evidencia descriptiva y exploratoria para orientar revisiones posteriores.

In [1]:
from copy import deepcopy
from pathlib import Path

import pandas as pd
import plotly.express as px

from plotly.subplots import make_subplots

import codigo.funciones_analisis as funciones

ORDEN_PLANTAS = [
    "Planta Centro",
    "Planta Norte",
    "Planta Sur"
]

COLORES_PLANTAS = {
    "Planta Centro": "#2E8B57",
    "Planta Norte": "#D55E00",
    "Planta Sur": "#0072B2"
}

COLOR_PROMEDIO = "#7F8C8D"
COLOR_MEDIANA = "#8E44AD"

In [2]:
RUTA_DATOS = (
    Path("datos")
    / "dataset_set_A_aguas_residuales.xlsx"
)

if not RUTA_DATOS.exists():
    raise FileNotFoundError(
        f"No se encontró el archivo: {RUTA_DATOS}"
    )

df_original = pd.read_excel(
    RUTA_DATOS
)

df = funciones.preparar_datos(
    df_original
)

print(
    f"Archivo cargado: {RUTA_DATOS}"
)

print(
    f"Filas originales: {df_original.shape[0]}"
)

print(
    f"Columnas originales: {df_original.shape[1]}"
)

print(
    f"Columnas preparadas: {df.shape[1]}"
)

df.head()

Archivo cargado: datos\dataset_set_A_aguas_residuales.xlsx
Filas originales: 200
Columnas originales: 10
Columnas preparadas: 12


,fecha_registro,planta,caudal_entrada_m3_d,DBO_entrada_mg_L,SST_entrada_mg_L,pH_entrada,energia_aeracion_kWh,lodos_generados_kg_d,DBO_salida_mg_L,cumplimiento_norma,eficiencia_remocion_DBO_pct,lodos_especificos_kg_m3
0,2025-08-17,Planta Sur,6562,271,324,7.12,1261.1,440.3,40.0,0,85.239852,0.067098
1,2025-09-07,Planta Sur,5336,322,230,7.01,1629.3,561.2,26.8,1,91.677019,0.105172
2,2025-07-26,Planta Norte,5755,318,282,6.79,1469.2,441.6,45.2,0,85.786164,0.076733
3,2025-10-27,Planta Centro,6840,216,167,7.01,1764.8,560.0,38.0,0,82.407407,0.081871
4,2025-09-06,Planta Centro,6803,326,200,7.55,1409.3,654.0,32.1,0,90.153374,0.096134


## 2. Inspección inicial del dataset

In [3]:
resumen_estructura = pd.DataFrame({
    "tipo_dato": df.dtypes.astype(str),
    "valores_no_nulos": df.notna().sum(),
    "valores_nulos": df.isna().sum(),
    "valores_unicos": df.nunique()
})

resumen_estructura

,tipo_dato,valores_no_nulos,valores_nulos,valores_unicos
fecha_registro,datetime64[us],200,0,98
planta,str,200,0,3
caudal_entrada_m3_d,int64,200,0,191
DBO_entrada_mg_L,int64,200,0,143
SST_entrada_mg_L,int64,200,0,138
pH_entrada,float64,200,0,109
energia_aeracion_kWh,float64,200,0,198
lodos_generados_kg_d,float64,200,0,199
DBO_salida_mg_L,float64,200,0,167
cumplimiento_norma,int64,200,0,2


In [4]:
print(
    "Tipo de dato:",
    df["fecha_registro"].dtype
)

print(
    "Fecha inicial:",
    df["fecha_registro"].min().date()
)

print(
    "Fecha final:",
    df["fecha_registro"].max().date()
)

Tipo de dato: datetime64[us]
Fecha inicial: 2025-07-01
Fecha final: 2025-10-28


In [5]:
auditoria_calidad = (
    funciones.auditar_calidad_datos(
        df_original
    )
)

auditoria_calidad["resumen_general"]

,indicador,valor
0,cantidad_filas,200
1,cantidad_columnas_originales,10
2,cantidad_columnas_preparadas,12
3,filas_duplicadas,0
4,combinaciones_fecha_planta_repetidas,38
5,max_registros_misma_fecha_planta,3
6,registros_adicionales_en_combinaciones,44
7,fecha_inicial,2025-07-01 00:00:00
8,fecha_final,2025-10-28 00:00:00
9,dias_del_periodo,120


In [6]:
auditoria_calidad["nulos_por_columna"]

,columna,cantidad_nulos
0,fecha_registro,0
1,planta,0
2,caudal_entrada_m3_d,0
3,DBO_entrada_mg_L,0
4,SST_entrada_mg_L,0
5,pH_entrada,0
6,energia_aeracion_kWh,0
7,lodos_generados_kg_d,0
8,DBO_salida_mg_L,0
9,cumplimiento_norma,0


In [7]:
auditoria_calidad[
    "combinaciones_fecha_planta_repetidas"
].head(10)

,fecha_registro,planta,cantidad_registros
4,2025-07-05,Planta Centro,2
8,2025-07-07,Planta Norte,2
11,2025-07-08,Planta Sur,2
12,2025-07-09,Planta Sur,2
16,2025-07-13,Planta Centro,2
26,2025-07-20,Planta Norte,2
30,2025-07-22,Planta Sur,2
31,2025-07-24,Planta Centro,2
32,2025-07-24,Planta Norte,2
37,2025-07-30,Planta Centro,3


### 2.1 Resultados de la inspección inicial

El dataset contiene 200 registros y 10 variables correspondientes a tres plantas. No se identificaron valores nulos ni filas completamente duplicadas. La información abarca desde el 1 de julio hasta el 28 de octubre de 2025, periodo de 120 días que comprende partes de cuatro meses calendario, aunque el caso lo presenta como el último trimestre.

Se identificaron 98 fechas únicas y 38 combinaciones fecha-planta con más de una observación. Estas combinaciones reúnen 44 registros adicionales posteriores al primero y alcanzan un máximo de tres observaciones para una misma planta y fecha. Los registros no son duplicados completos, pero el dataset no incluye hora ni identificador de muestra. Esta limitación impide establecer el orden intradía y exige resumir con cautela la información temporal, sin eliminar observaciones cuya diferencia podría ser válida.

In [8]:
auditoria_calidad["controles_consistencia"]

,control,cantidad_registros
0,caudal no positivo,0
1,DBO de entrada no positiva,0
2,SST de entrada no positivo,0
3,pH fuera del rango físico de 0 a 14,0
4,DBO de salida negativa,0
5,energía de aireación negativa,0
6,lodos generados negativos,0
7,DBO de salida mayor que DBO de entrada,0
8,cumplimiento fuera de 0 y 1,0


In [9]:
auditoria_calidad["resumen_atipicos"]

,grupo_analisis,variable,q1,q3,rango_intercuartil,limite_inferior,limite_superior,cantidad_candidatos
0,Planta Centro,caudal_entrada_m3_d,4320.5000,6194.0000,1873.500,1510.2500,9004.2500,1
1,Planta Centro,DBO_entrada_mg_L,232.0000,326.0000,94.000,91.0000,467.0000,1
2,Planta Centro,SST_entrada_mg_L,199.5000,276.0000,76.500,84.7500,390.7500,0
3,Planta Centro,pH_entrada,6.9200,7.4700,0.550,6.0950,8.2950,0
4,Planta Centro,energia_aeracion_kWh,962.7000,1537.5000,574.800,100.5000,2399.7000,0
5,Planta Centro,lodos_generados_kg_d,330.7000,538.4000,207.700,19.1500,849.9500,0
6,Planta Centro,DBO_salida_mg_L,26.2500,46.2000,19.950,-3.6750,76.1250,0
7,Planta Norte,caudal_entrada_m3_d,4365.0000,6188.0000,1823.000,1630.5000,8922.5000,2
8,Planta Norte,DBO_entrada_mg_L,212.5000,330.5000,118.000,35.5000,507.5000,0
9,Planta Norte,SST_entrada_mg_L,182.5000,265.0000,82.500,58.7500,388.7500,1


In [10]:
auditoria_calidad["candidatos_atipicos"]

,fecha_registro,planta,caudal_entrada_m3_d,DBO_entrada_mg_L,SST_entrada_mg_L,pH_entrada,energia_aeracion_kWh,lodos_generados_kg_d,DBO_salida_mg_L,cumplimiento_norma,eficiencia_remocion_DBO_pct,lodos_especificos_kg_m3,grupo_analisis,variable,valor,limite_inferior,limite_superior
0,2025-08-11,Planta Centro,1500,381,189,7.37,279.7,117.1,55.0,0,85.564304,0.078067,Planta Centro,caudal_entrada_m3_d,1500.0,1510.250,9004.250
1,2025-07-13,Planta Centro,4283,481,179,7.61,741.9,294.5,48.1,0,90.000000,0.068760,Planta Centro,DBO_entrada_mg_L,481.0,91.000,467.000
2,2025-07-20,Planta Norte,9205,196,207,6.54,2434.0,971.7,36.5,0,81.377551,0.105562,Planta Norte,caudal_entrada_m3_d,9205.0,1630.500,8922.500
3,2025-09-30,Planta Norte,1500,294,231,7.50,383.8,113.7,45.4,0,84.557823,0.075800,Planta Norte,caudal_entrada_m3_d,1500.0,1630.500,8922.500
4,2025-07-20,Planta Norte,5634,133,391,7.46,1135.5,597.4,24.7,0,81.428571,0.106035,Planta Norte,SST_entrada_mg_L,391.0,58.750,388.750
5,2025-07-20,Planta Norte,9205,196,207,6.54,2434.0,971.7,36.5,0,81.377551,0.105562,Planta Norte,energia_aeracion_kWh,2434.0,402.875,2174.275
6,2025-09-11,Planta Norte,1992,214,165,7.56,388.6,195.8,26.4,1,87.663551,0.098293,Planta Norte,energia_aeracion_kWh,388.6,402.875,2174.275
7,2025-09-30,Planta Norte,1500,294,231,7.50,383.8,113.7,45.4,0,84.557823,0.075800,Planta Norte,energia_aeracion_kWh,383.8,402.875,2174.275
8,2025-07-20,Planta Norte,9205,196,207,6.54,2434.0,971.7,36.5,0,81.377551,0.105562,Planta Norte,lodos_generados_kg_d,971.7,186.150,731.750
9,2025-09-30,Planta Norte,1500,294,231,7.50,383.8,113.7,45.4,0,84.557823,0.075800,Planta Norte,lodos_generados_kg_d,113.7,186.150,731.750


El criterio del rango intercuartílico, aplicado por planta a siete variables numéricas, identificó 15 casos candidatos a atípicos, correspondientes a 10 registros distintos. Se conservaron porque el criterio no demuestra que exista un error de medición.

## 3. Construcción de variables derivadas

Para comparar el desempeño observado se calculará la eficiencia de remoción de DBO como la diferencia entre la concentración de entrada y salida, dividida por la concentración de entrada. También se calculará la generación específica de lodos mediante la relación entre los kilogramos diarios de lodos y el caudal tratado diariamente.

No se calculará energía específica porque el nombre de la variable `energia_aeracion_kWh` no identifica el periodo asociado al consumo. Sin esta información no es posible asegurar que su división por el caudal diario produzca una medida válida en kWh/m³.

Las variables derivadas se utilizarán con fines descriptivos y comparativos. Sus resultados no establecerán por sí solos que una planta funciona correctamente o presenta una deficiencia, debido a la ausencia de metas operacionales, capacidades de diseño y rangos técnicos de referencia.

In [11]:
columnas_derivadas = [
    "fecha_registro",
    "planta",
    "DBO_entrada_mg_L",
    "DBO_salida_mg_L",
    "eficiencia_remocion_DBO_pct",
    "lodos_especificos_kg_m3"
]

columnas_faltantes = [
    columna
    for columna in columnas_derivadas
    if columna not in df.columns
]

if columnas_faltantes:
    raise ValueError(
        "Faltan variables preparadas: "
        + ", ".join(columnas_faltantes)
    )

vista_variables_derivadas = (
    df[columnas_derivadas]
    .head()
    .copy()
)

columnas_numericas_derivadas = [
    "DBO_entrada_mg_L",
    "DBO_salida_mg_L",
    "eficiencia_remocion_DBO_pct",
    "lodos_especificos_kg_m3"
]

vista_variables_derivadas[
    columnas_numericas_derivadas
] = vista_variables_derivadas[
    columnas_numericas_derivadas
].round(2)

vista_variables_derivadas

,fecha_registro,planta,DBO_entrada_mg_L,DBO_salida_mg_L,eficiencia_remocion_DBO_pct,lodos_especificos_kg_m3
0,2025-08-17,Planta Sur,271,40.0,85.24,0.07
1,2025-09-07,Planta Sur,322,26.8,91.68,0.11
2,2025-07-26,Planta Norte,318,45.2,85.79,0.08
3,2025-10-27,Planta Centro,216,38.0,82.41,0.08
4,2025-09-06,Planta Centro,326,32.1,90.15,0.10


## 4. Análisis descriptivo y comparativo por planta

La comparación se realizará considerando la cantidad de observaciones de cada planta, medidas de tendencia central y variables operacionales y ambientales. Se utilizarán el promedio y la mediana para describir el comportamiento central, junto con medidas de dispersión y visualizaciones que permitan identificar variabilidad y posibles valores atípicos.

In [12]:
resumen_resultados = (
    funciones.crear_resumen_resultados(
        df_original
    )
)

resumen_por_planta = (
    resumen_resultados["resumen_por_planta"]
)

resumen_por_planta.round(2)

,planta,registros,caudal_promedio_m3_d,DBO_salida_promedio_mg_L,DBO_salida_mediana_mg_L,eficiencia_promedio_pct,eficiencia_mediana_pct,cumplimiento_registrado_pct,lodos_especificos_mediana_kg_m3
0,Planta Centro,75,5112.72,35.90,35.50,87.51,87.76,22.67,0.09
1,Planta Norte,71,5287.87,36.56,34.20,86.65,86.72,16.90,0.09
2,Planta Sur,54,4684.52,36.06,34.65,87.10,86.73,29.63,0.08


### 4.1 Comparación inicial

La cantidad de registros no es igual entre plantas: Centro contiene 75 observaciones, Norte 71 y Sur 54. Por esta razón, las comparaciones se basan en promedios, medianas y proporciones, mientras que los tamaños muestrales se conservan para interpretar el respaldo de cada estimación.

Las eficiencias promedio de remoción son cercanas. Planta Centro presenta el mayor valor observado, con 87,51 %, y Planta Norte el menor, con 86,65 %. Esta diferencia es inferior a un punto porcentual y no permite afirmar por sí sola que una instalación funciona mejor que otra.

Las tres plantas registran una DBO de salida promedio próxima a 36 mg/L. Sin embargo, las proporciones de cumplimiento registrado varían desde 16,90 % en Planta Norte hasta 29,63 % en Planta Sur. Estos resúmenes no permiten reconstruir la regla utilizada para clasificar el cumplimiento, por lo que esta variable se analizará como una etiqueta incluida en el dataset y no como una verificación normativa.

In [13]:
comparacion_eficiencia = resumen_por_planta.melt(
    id_vars="planta",
    value_vars=[
        "eficiencia_promedio_pct",
        "eficiencia_mediana_pct"
    ],
    var_name="medida",
    value_name="eficiencia_pct"
)

comparacion_eficiencia["medida"] = (
    comparacion_eficiencia["medida"]
    .replace({
        "eficiencia_promedio_pct": "Promedio",
        "eficiencia_mediana_pct": "Mediana"
    })
)

fig_eficiencia = px.bar(
    comparacion_eficiencia,
    x="planta",
    y="eficiencia_pct",
    color="medida",
    category_orders={
        "planta": ORDEN_PLANTAS,
        "medida": ["Promedio", "Mediana"]
    },
    barmode="group",
    text_auto=".2f",
    title="Eficiencia de remoción de DBO por planta",
    labels={
        "planta": "Planta",
        "eficiencia_pct": "Eficiencia de remoción (%)",
        "medida": "Medida"
    },
    color_discrete_map={
        "Promedio": COLOR_PROMEDIO,
        "Mediana": COLOR_MEDIANA
    },
    template="plotly_dark"
)

fig_eficiencia.update_traces(
    textposition="outside",
    cliponaxis=False
)

fig_eficiencia.update_yaxes(
    range=[0, 100]
)

fig_eficiencia.update_layout(
    height=500,
    legend_title_text=""
)

fig_eficiencia.show()

In [14]:
fig_distribucion_eficiencia = px.box(
    df,
    x="planta",
    y="eficiencia_remocion_DBO_pct",
    color="planta",
    category_orders={
        "planta": ORDEN_PLANTAS
    },
    color_discrete_map=COLORES_PLANTAS,
    points="outliers",
    title="Distribución de la eficiencia de remoción de DBO",
    labels={
        "planta": "Planta",
        "eficiencia_remocion_DBO_pct":
            "Eficiencia de remoción (%)"
    },
    hover_data={
        "fecha_registro": True,
        "DBO_entrada_mg_L": ":.1f",
        "DBO_salida_mg_L": ":.1f"
    },
    template="plotly_dark"
)

fig_distribucion_eficiencia.update_layout(
    height=500,
    showlegend=False
)

fig_distribucion_eficiencia.show()

In [15]:
eficiencia_diaria = (
    df.groupby(
        ["fecha_registro", "planta"],
        as_index=False
    )
    .agg(
        eficiencia_mediana_pct=(
            "eficiencia_remocion_DBO_pct",
            "median"
        ),
        cantidad_registros=(
            "eficiencia_remocion_DBO_pct",
            "size"
        )
    )
)

fig_eficiencia_temporal = px.line(
    eficiencia_diaria,
    x="fecha_registro",
    y="eficiencia_mediana_pct",
    color="planta",
    category_orders={
        "planta": ORDEN_PLANTAS
    },
    color_discrete_map=COLORES_PLANTAS,
    markers=True,
    title="Evolución de la eficiencia mediana de remoción por fecha",
    labels={
        "fecha_registro": "Fecha",
        "eficiencia_mediana_pct": "Eficiencia mediana (%)",
        "planta": "Planta"
    },
    hover_data={
        "cantidad_registros": True
    },
    template="plotly_dark"
)

fig_eficiencia_temporal.update_layout(
    height=550,
    legend_title_text=""
)

fig_eficiencia_temporal.update_xaxes(
    tickformat="%d-%m-%Y"
)

fig_eficiencia_temporal.show()

In [16]:
clasificaciones_por_DBO = (
    df.groupby("DBO_salida_mg_L")["cumplimiento_norma"]
      .nunique()
)

valores_con_clasificacion_mixta = (
    clasificaciones_por_DBO[
        clasificaciones_por_DBO > 1
    ]
    .index
    .tolist()
)

registros_clasificacion_mixta = (
    df[
        df["DBO_salida_mg_L"].isin(
            valores_con_clasificacion_mixta
        )
    ]
    [
        [
            "fecha_registro",
            "planta",
            "DBO_salida_mg_L",
            "cumplimiento_norma"
        ]
    ]
    .sort_values(
        ["DBO_salida_mg_L", "cumplimiento_norma"]
    )
)

print(
    "Valores de DBO de salida presentes en ambas clasificaciones:",
    valores_con_clasificacion_mixta
)

registros_clasificacion_mixta

Valores de DBO de salida presentes en ambas clasificaciones: [21.3, 24.7, 25.4, 25.6, 26.4]


,fecha_registro,planta,DBO_salida_mg_L,cumplimiento_norma
11,2025-07-24,Planta Norte,21.3,0
36,2025-08-23,Planta Norte,21.3,1
81,2025-07-20,Planta Norte,24.7,0
65,2025-10-14,Planta Norte,24.7,1
143,2025-09-30,Planta Centro,25.4,0
66,2025-08-05,Planta Sur,25.4,1
129,2025-08-09,Planta Centro,25.6,0
142,2025-08-14,Planta Centro,25.6,1
96,2025-07-19,Planta Centro,26.4,0
15,2025-08-12,Planta Centro,26.4,1


### 4.2 Cumplimiento registrado y DBO de salida

La variable `cumplimiento_norma` se analizará como una clasificación proporcionada por el dataset. No se interpretará como una verificación legal, porque no se dispone de la norma aplicada, los criterios complementarios ni la metodología utilizada para asignar cada etiqueta.

Se identificaron cinco valores de DBO de salida presentes tanto en registros clasificados con cumplimiento como en registros clasificados con incumplimiento: 21,3; 24,7; 25,4; 25,6 y 26,4 mg/L. Este hallazgo no demuestra que las etiquetas sean erróneas, pero confirma que la concentración de salida no permite explicar por sí sola la clasificación registrada. Por esta razón, los resultados se presentarán sin sustituir, corregir ni reinterpretar las etiquetas originales.

In [17]:
fig_cumplimiento = px.bar(
    resumen_por_planta,
    x="planta",
    y="cumplimiento_registrado_pct",
    color="planta",
    category_orders={
        "planta": ORDEN_PLANTAS
    },
    color_discrete_map=COLORES_PLANTAS,
    text_auto=".2f",
    title="Proporción de cumplimiento registrado por planta",
    labels={
        "planta": "Planta",
        "cumplimiento_registrado_pct":
            "Cumplimiento registrado (%)"
    },
    template="plotly_dark"
)

fig_cumplimiento.update_traces(
    textposition="outside",
    cliponaxis=False
)

fig_cumplimiento.update_yaxes(
    range=[0, 100]
)

fig_cumplimiento.update_layout(
    height=500,
    showlegend=False
)

fig_cumplimiento.show()

In [18]:
fig_caudal_energia = px.scatter(
    df,
    x="caudal_entrada_m3_d",
    y="energia_aeracion_kWh",
    color="planta",
    category_orders={
        "planta": ORDEN_PLANTAS
    },
    color_discrete_map=COLORES_PLANTAS,
    opacity=0.75,
    title="Relación entre caudal de entrada y energía de aireación",
    labels={
        "caudal_entrada_m3_d": "Caudal de entrada (m³/día)",
        "energia_aeracion_kWh":
            "Energía de aireación registrada (kWh)",
        "planta": "Planta"
    },
    hover_data={
        "fecha_registro": True,
        "caudal_entrada_m3_d": ":.0f",
        "energia_aeracion_kWh": ":.1f"
    },
    template="plotly_dark"
)

fig_caudal_energia.update_layout(
    height=550,
    legend_title_text=""
)

fig_caudal_energia.show()

In [19]:
correlacion_caudal_energia = (
    funciones.calcular_correlaciones_por_planta(
        df,
        "caudal_entrada_m3_d",
        "energia_aeracion_kWh"
    )
)

tabla_correlacion_caudal_energia = (
    correlacion_caudal_energia.copy()
)

tabla_correlacion_caudal_energia[
    ["pearson", "spearman"]
] = tabla_correlacion_caudal_energia[
    ["pearson", "spearman"]
].round(3)

for columna in [
    "pearson_valor_p",
    "spearman_valor_p"
]:
    tabla_correlacion_caudal_energia[columna] = (
        tabla_correlacion_caudal_energia[columna]
        .map("{:.3e}".format)
    )

tabla_correlacion_caudal_energia

,planta,observaciones,pearson,pearson_valor_p,spearman,spearman_valor_p
0,Planta Centro,75,0.832,2.526e-20,0.869,5.832e-24
1,Planta Norte,71,0.863,4.284e-22,0.835,1.390e-19
2,Planta Sur,54,0.873,7.991e-18,0.838,2.629e-15


### 4.3 Relación entre caudal y energía de aireación

En esta muestra se observa una asociación positiva fuerte entre el caudal de entrada y la energía de aireación registrada en las tres plantas. Los coeficientes de Pearson se encuentran entre 0,832 y 0,873, mientras que los coeficientes de Spearman varían entre 0,835 y 0,869.

Estos resultados indican que los registros con mayor caudal tienden a presentar valores superiores de energía de aireación. Sin embargo, la asociación no demuestra que el aumento del caudal sea la única causa del consumo energético ni permite comparar eficiencia entre plantas. Además, el dataset no especifica el periodo asociado a la variable de energía, por lo que no se calculará un indicador en kWh/m³ sin validación adicional. Los valores p deben interpretarse con cautela, porque no puede garantizarse la independencia entre observaciones registradas para una misma planta y fecha.

In [20]:
correlacion_DBO_entrada_salida = (
    funciones.calcular_correlaciones_por_planta(
        df,
        "DBO_entrada_mg_L",
        "DBO_salida_mg_L"
    )
)

correlacion_DBO_entrada_salida.insert(
    1,
    "relacion",
    "DBO de entrada - DBO de salida"
)

correlacion_caudal_DBO_salida = (
    funciones.calcular_correlaciones_por_planta(
        df,
        "caudal_entrada_m3_d",
        "DBO_salida_mg_L"
    )
)

correlacion_caudal_DBO_salida.insert(
    1,
    "relacion",
    "Caudal de entrada - DBO de salida"
)

correlaciones_DBO_salida = pd.concat(
    [
        correlacion_DBO_entrada_salida,
        correlacion_caudal_DBO_salida
    ],
    ignore_index=True
)

tabla_correlaciones_DBO_salida = (
    correlaciones_DBO_salida.copy()
)

tabla_correlaciones_DBO_salida[
    ["pearson", "spearman"]
] = tabla_correlaciones_DBO_salida[
    ["pearson", "spearman"]
].round(3)

for columna in [
    "pearson_valor_p",
    "spearman_valor_p"
]:
    tabla_correlaciones_DBO_salida[columna] = (
        tabla_correlaciones_DBO_salida[columna]
        .map("{:.3e}".format)
    )

tabla_correlaciones_DBO_salida

,planta,relacion,observaciones,pearson,pearson_valor_p,spearman,spearman_valor_p
0,Planta Centro,DBO de entrada - DBO de salida,75,0.734,7.175e-14,0.726,1.767e-13
1,Planta Norte,DBO de entrada - DBO de salida,71,0.750,5.212e-14,0.719,1.633e-12
2,Planta Sur,DBO de entrada - DBO de salida,54,0.817,4.753e-14,0.820,3.396e-14
3,Planta Centro,Caudal de entrada - DBO de salida,75,-0.059,6.124e-01,-0.046,6.924e-01
4,Planta Norte,Caudal de entrada - DBO de salida,71,0.135,2.620e-01,0.169,1.588e-01
5,Planta Sur,Caudal de entrada - DBO de salida,54,0.295,3.019e-02,0.281,3.947e-02


### 4.4 Relación de la DBO de salida con la entrada y el caudal

La concentración de DBO de entrada presentó una asociación positiva con la DBO de salida en las tres plantas. Los coeficientes de Pearson variaron entre 0,734 y 0,817, mientras que los coeficientes de Spearman se situaron entre 0,719 y 0,820. Los valores p fueron inferiores a 0,001 en todos los casos, por lo que la asociación observada es estadísticamente significativa en esta muestra. Este resultado indica que concentraciones mayores de DBO de entrada tienden a coincidir con concentraciones mayores de salida, pero no demuestra causalidad.

La relación entre el caudal de entrada y la DBO de salida no presentó un patrón consistente. En Planta Centro la asociación fue prácticamente nula; en Planta Norte fue positiva débil y no significativa; y en Planta Sur fue positiva débil, con valores p inferiores a 0,05. Por tanto, la evidencia respalda una relación entre las concentraciones de DBO de entrada y salida, pero no permite sostener una asociación general entre el caudal y la DBO de salida para las tres plantas. Los valores p deben interpretarse con cautela, porque no puede garantizarse la independencia entre observaciones registradas para una misma planta y fecha.

In [21]:
fig_lodos_especificos = px.box(
    df,
    x="planta",
    y="lodos_especificos_kg_m3",
    color="planta",
    category_orders={
        "planta": ORDEN_PLANTAS
    },
    color_discrete_map=COLORES_PLANTAS,
    points="outliers",
    title="Distribución de la generación específica de lodos",
    labels={
        "planta": "Planta",
        "lodos_especificos_kg_m3":
            "Generación específica de lodos (kg/m³)"
    },
    hover_data={
        "fecha_registro": True,
        "caudal_entrada_m3_d": ":.0f",
        "lodos_generados_kg_d": ":.1f"
    },
    template="plotly_dark"
)

fig_lodos_especificos.update_layout(
    height=500,
    showlegend=False
)

fig_lodos_especificos.show()

In [22]:
fig_DBO_salida = px.box(
    df,
    x="planta",
    y="DBO_salida_mg_L",
    color="planta",
    category_orders={
        "planta": ORDEN_PLANTAS
    },
    color_discrete_map=COLORES_PLANTAS,
    points="outliers",
    title="Distribución de la DBO de salida por planta",
    labels={
        "planta": "Planta",
        "DBO_salida_mg_L": "DBO de salida (mg/L)"
    },
    hover_data={
        "fecha_registro": True,
        "DBO_entrada_mg_L": ":.1f",
        "cumplimiento_norma": True
    },
    template="plotly_dark"
)

fig_DBO_salida.update_layout(
    height=500,
    showlegend=False
)

fig_DBO_salida.show()

In [23]:
resultado_atipicos_DBO = (
    funciones.detectar_candidatos_atipicos_iqr(
        df,
        ["DBO_salida_mg_L"]
    )
)

candidatos_DBO_salida = (
    resultado_atipicos_DBO["candidatos"]
)

columnas_revision_DBO = [
    "fecha_registro",
    "planta",
    "DBO_entrada_mg_L",
    "DBO_salida_mg_L",
    "cumplimiento_norma",
    "limite_inferior",
    "limite_superior"
]

print(
    "Cantidad de candidatos atípicos:",
    len(candidatos_DBO_salida)
)

tabla_candidatos_DBO = candidatos_DBO_salida[
    columnas_revision_DBO
].copy()

columnas_numericas_revision = [
    "DBO_entrada_mg_L",
    "DBO_salida_mg_L",
    "limite_inferior",
    "limite_superior"
]

tabla_candidatos_DBO[
    columnas_numericas_revision
] = tabla_candidatos_DBO[
    columnas_numericas_revision
].round(2)

tabla_candidatos_DBO

Cantidad de candidatos atípicos: 2


,fecha_registro,planta,DBO_entrada_mg_L,DBO_salida_mg_L,cumplimiento_norma,limite_inferior,limite_superior
0,2025-07-01,Planta Norte,432,79.0,0,-3.45,74.15
1,2025-10-25,Planta Norte,432,76.8,0,-3.45,74.15


### 4.5 Revisión de posibles valores atípicos

El criterio del rango intercuartílico identificó dos candidatos atípicos en la DBO de salida de Planta Norte. Los registros corresponden al 1 de julio y al 25 de octubre de 2025, con concentraciones de 79,0 y 76,8 mg/L, respectivamente. Ambos valores superan el límite estadístico superior de 74,15 mg/L calculado para esa planta.

Estas observaciones no presentan valores nulos, no corresponden a filas duplicadas y mantienen una DBO de salida inferior a la concentración de entrada. Por esta razón, se conservarán en el análisis. El criterio estadístico permite identificar registros que requieren revisión, pero no demuestra que exista un error de medición ni justifica su eliminación automática.

## 5. Dashboard exploratorio

El dashboard integra los principales resultados operacionales y ambientales para facilitar la comparación entre plantas. Incluye la eficiencia de remoción de DBO, la concentración de salida, el cumplimiento registrado, la relación entre caudal y energía de aireación y la evolución temporal de los indicadores.

Las visualizaciones conservan el detalle disponible mediante información emergente y herramientas de exploración. Los resultados describen el comportamiento de la muestra y deben interpretarse junto con las limitaciones de calidad, granularidad y contexto técnico identificadas durante el análisis.

In [24]:
dashboard = make_subplots(
    rows=3,
    cols=2,
    specs=[
        [{}, {}],
        [{}, {}],
        [{"colspan": 2}, None]
    ],
    subplot_titles=[
        "Eficiencia promedio y mediana",
        "Cumplimiento registrado",
        "Caudal y energía de aireación",
        "Distribución de la DBO de salida",
        "Evolución temporal de la eficiencia"
    ],
    vertical_spacing=0.10,
    horizontal_spacing=0.10
)


def agregar_trazas(
    figura_origen,
    fila,
    columna,
    mostrar_leyenda
):
    for traza in figura_origen.data:
        nueva_traza = deepcopy(traza)
        nueva_traza.showlegend = mostrar_leyenda

        dashboard.add_trace(
            nueva_traza,
            row=fila,
            col=columna
        )


agregar_trazas(
    fig_eficiencia,
    fila=1,
    columna=1,
    mostrar_leyenda=True
)

agregar_trazas(
    fig_cumplimiento,
    fila=1,
    columna=2,
    mostrar_leyenda=False
)

agregar_trazas(
    fig_caudal_energia,
    fila=2,
    columna=1,
    mostrar_leyenda=True
)

agregar_trazas(
    fig_DBO_salida,
    fila=2,
    columna=2,
    mostrar_leyenda=False
)

agregar_trazas(
    fig_eficiencia_temporal,
    fila=3,
    columna=1,
    mostrar_leyenda=False
)

dashboard.update_yaxes(
    title_text="Eficiencia (%)",
    range=[0, 100],
    row=1,
    col=1
)

dashboard.update_yaxes(
    title_text="Cumplimiento registrado (%)",
    range=[0, 100],
    row=1,
    col=2
)

dashboard.update_xaxes(
    title_text="Caudal de entrada (m³/día)",
    row=2,
    col=1
)

dashboard.update_yaxes(
    title_text="Energía registrada (kWh)",
    row=2,
    col=1
)

dashboard.update_yaxes(
    title_text="DBO de salida (mg/L)",
    row=2,
    col=2
)

dashboard.update_xaxes(
    title_text="Fecha",
    tickformat="%d-%m-%Y",
    rangeslider_visible=True,
    row=3,
    col=1
)

dashboard.update_yaxes(
    title_text="Eficiencia mediana (%)",
    row=3,
    col=1
)

dashboard.update_layout(
    title={
        "text": "Dashboard exploratorio de AquaLimpia S. A.",
        "x": 0.5,
        "xanchor": "center"
    },
    template="plotly_dark",
    height=1250,
    barmode="group",
    boxmode="group",
    hovermode="closest",
    legend={
        "orientation": "h",
        "yanchor": "bottom",
        "y": 1.04,
        "xanchor": "center",
        "x": 0.5
    },
    margin={
        "t": 140,
        "b": 80,
        "l": 70,
        "r": 40
    }
)

dashboard.show()

In [25]:
RUTA_RESULTADOS = Path("resultados")
RUTA_RESULTADOS.mkdir(exist_ok=True)

RUTA_DASHBOARD = (
    RUTA_RESULTADOS
    / "dashboard_aqualimpia.html"
)

dashboard.write_html(
    RUTA_DASHBOARD,
    include_plotlyjs=True,
    full_html=True
)

print(
    "Dashboard guardado en:",
    RUTA_DASHBOARD
)

Dashboard guardado en: resultados\dashboard_aqualimpia.html


In [26]:
reporte_operaciones = (
    funciones.crear_reporte_operaciones(
        df_original
    )
)

reporte_gestion_ambiental = (
    funciones.crear_reporte_gestion_ambiental(
        df_original
    )
)

print(
    "Reporte de Operaciones:",
    reporte_operaciones.shape
)

print(
    "Reporte de Gestión Ambiental:",
    reporte_gestion_ambiental.shape
)

display(
    reporte_operaciones.head(3)
)

display(
    reporte_gestion_ambiental.head(3)
)

Reporte de Operaciones: (200, 9)
Reporte de Gestión Ambiental: (200, 5)


,fecha_registro,planta,caudal_entrada_m3_d,DBO_entrada_mg_L,DBO_salida_mg_L,eficiencia_remocion_DBO_pct,energia_aeracion_kWh,lodos_generados_kg_d,lodos_especificos_kg_m3
0,2025-07-01,Planta Norte,4545,432,79.0,81.712963,1450.1,399.3,0.087855
1,2025-07-02,Planta Norte,5860,235,32.0,86.382979,1528.4,494.0,0.084300
2,2025-07-03,Planta Centro,5124,396,64.9,83.611111,1035.3,459.8,0.089735


,fecha_registro,planta,DBO_salida_mg_L,cumplimiento_norma,cumplimiento_registrado
0,2025-07-01,Planta Norte,79.0,0,Incumplimiento registrado
1,2025-07-02,Planta Norte,32.0,0,Incumplimiento registrado
2,2025-07-03,Planta Centro,64.9,0,Incumplimiento registrado


In [27]:
rutas_reportes = (
    funciones.guardar_reportes_excel(
        df_original,
        RUTA_RESULTADOS
    )
)

for nombre, ruta in rutas_reportes.items():
    print(
        f"{nombre}: {ruta} | "
        f"existe: {ruta.exists()}"
    )

operaciones: resultados\reporte_operaciones.xlsx | existe: True
gestion_ambiental: resultados\reporte_gestion_ambiental.xlsx | existe: True


In [28]:
reporte_operaciones_verificado = pd.read_excel(
    rutas_reportes["operaciones"]
)

reporte_ambiental_verificado = pd.read_excel(
    rutas_reportes["gestion_ambiental"]
)

pd.testing.assert_frame_equal(
    reporte_operaciones_verificado,
    reporte_operaciones,
    check_dtype=False,
    check_exact=False,
    rtol=1e-10
)

pd.testing.assert_frame_equal(
    reporte_ambiental_verificado,
    reporte_gestion_ambiental,
    check_dtype=False,
    check_exact=False,
    rtol=1e-10
)

print(
    "Reportes verificados correctamente."
)

Reportes verificados correctamente.


In [29]:
RUTA_RESUMEN_JOBLIB = (
    RUTA_RESULTADOS
    / "resumen_resultados.joblib"
)

ruta_resumen_guardado = (
    funciones.guardar_resumen_joblib(
        df_original,
        RUTA_RESUMEN_JOBLIB
    )
)

resumen_esperado = (
    funciones.crear_resumen_resultados(
        df_original
    )
)

resumen_recuperado = (
    funciones.cargar_resumen_joblib(
        RUTA_RESUMEN_JOBLIB
    )
)

pd.testing.assert_frame_equal(
    resumen_recuperado["resumen_por_planta"],
    resumen_esperado["resumen_por_planta"]
)

for clave in [
    "cantidad_registros",
    "cantidad_plantas",
    "fecha_inicial",
    "fecha_final"
]:
    assert (
        resumen_recuperado[clave]
        == resumen_esperado[clave]
    )

print(
    "Archivo guardado:",
    ruta_resumen_guardado
)

print(
    "Existe:",
    ruta_resumen_guardado.exists()
)

print(
    "Claves recuperadas:",
    list(resumen_recuperado.keys())
)

display(
    resumen_recuperado[
        "resumen_por_planta"
    ].round(2)
)

Archivo guardado: resultados\resumen_resultados.joblib
Existe: True
Claves recuperadas: ['cantidad_registros', 'cantidad_plantas', 'fecha_inicial', 'fecha_final', 'resumen_por_planta']


,planta,registros,caudal_promedio_m3_d,DBO_salida_promedio_mg_L,DBO_salida_mediana_mg_L,eficiencia_promedio_pct,eficiencia_mediana_pct,cumplimiento_registrado_pct,lodos_especificos_mediana_kg_m3
0,Planta Centro,75,5112.72,35.90,35.50,87.51,87.76,22.67,0.09
1,Planta Norte,71,5287.87,36.56,34.20,86.65,86.72,16.90,0.09
2,Planta Sur,54,4684.52,36.06,34.65,87.10,86.73,29.63,0.08


### 5.1 Persistencia de resultados con Joblib

El archivo `resultados/resumen_resultados.joblib` conserva los principales resultados calculados, incluido el periodo analizado, la cantidad de registros, el número de plantas y el resumen comparativo por instalación. Su propósito es permitir la recuperación de estos objetos sin repetir el procesamiento completo.

Joblib no reemplaza los reportes Excel ni el dashboard, porque su contenido está destinado a reutilización programática y no a lectura directa. Por seguridad, el archivo solo debe cargarse cuando proviene de una fuente confiable, ya que los formatos de serialización de objetos pueden ejecutar contenido malicioso durante su lectura.

La prueba realizada guardó el resumen, lo recuperó desde el archivo y comparó sus valores con el objeto original. No se detectaron diferencias.

## 6. Reflexión final

El desarrollo permitió comprobar que una solución analítica reproducible no depende únicamente de generar gráficos, sino también de validar la calidad de los datos, modularizar los cálculos y documentar sus límites. NumPy permitió aplicar de forma reutilizable el criterio del rango intercuartílico, SciPy respaldó las asociaciones mediante coeficientes y valores p, y Joblib permitió persistir y recuperar el resumen de resultados.

El análisis también mostró la importancia de distinguir entre evidencia estadística e interpretación operacional. Un candidato atípico no constituye por sí solo un error, una correlación no demuestra causalidad y una etiqueta de cumplimiento sin criterio documentado no permite verificar una norma. Estas precauciones evitan presentar conclusiones más amplias que los datos disponibles.

Aunque el trabajo fue individual, el uso de funciones externas, rutas relativas, reportes diferenciados y control de versiones mejora la trazabilidad y deja una base preparada para revisión, reutilización y colaboración futura. Como mejora posterior, sería necesario incorporar variables del proceso, criterios normativos documentados y una cobertura temporal más amplia antes de evaluar modelos predictivos o formular recomendaciones operacionales.